# Home Credit — Data Analysis, Feature Alignment & Model Evaluation

Notebook này được cập nhật theo pipeline feature/model hiện tại:
1. **Setup** — kết nối DuckDB, đọc contract feature từ training scripts
2. **Đánh giá chất lượng dữ liệu** — class balance, null rate, correlation trên Gold v3
3. **EDA** — default rate theo các feature đang dùng: credit score, DTI, years employed, occupation
4. **Train models** — LightGBM customer risk v3 + LR Scorecard
5. **Đánh giá model** — ROC, confusion matrix, feature importance, score distribution, calibration, Precision-Recall

> Lưu ý: `ext_source_1` và `ext_source_3` đã bị loại khỏi model v3 vì không có tại inference. Thay vào đó model dùng `years_employed` và `occupation_type`, cùng các feature payload/bureau/history/demographic bắt buộc.


## 1. Setup & Overview

In [ ]:
import sys
from pathlib import Path

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path('/home/taitu/GitHub/Loan_ETL')]
    for candidate in candidates:
        if (candidate / 'machinelearning' / 'ml' / 'retrain_customer_model.py').exists():
            return candidate / 'machinelearning'
        if (candidate / 'data' / 'etl.duckdb').exists() or (candidate / 'ml' / 'retrain_customer_model.py').exists():
            return candidate
    return cwd

ROOT = find_project_root()
PROJECT_ROOT = ROOT.parent if ROOT.name == 'machinelearning' else ROOT
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from machinelearning.ml.retrain_customer_model import (
    ALL_FEATURES as LGBM_FEATURES,
    NUMERIC_FEATURES as LGBM_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES as LGBM_CATEGORICAL_FEATURES,
    MODEL_VERSION as LGBM_MODEL_VERSION,
)
from machinelearning.ml.train_scorecard import (
    ALL_FEATURES as SCORECARD_FEATURES,
    NUMERIC_FEATURES as SCORECARD_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES as SCORECARD_CATEGORICAL_FEATURES,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

con = duckdb.connect(str(ROOT / 'data' / 'etl.duckdb'), read_only=True)

tables = con.execute('''
SELECT 
    t.table_schema as schema,
    t.table_name as name,
    COUNT(c.column_name) as column_count
FROM information_schema.tables t
LEFT JOIN information_schema.columns c 
    ON t.table_schema = c.table_schema 
    AND t.table_name = c.table_name
WHERE t.table_schema NOT IN ('information_schema', 'system')
GROUP BY t.table_schema, t.table_name
ORDER BY schema, name
''').df()

tables['row_count'] = tables.apply(
    lambda r: con.execute(f"SELECT COUNT(*) FROM {r['schema']}.{r['name']}").fetchone()[0],
    axis=1
)

print('Connected ✓')
print(f'LightGBM artifact contract: {LGBM_MODEL_VERSION} | {len(LGBM_FEATURES)} features')
print(f'  numeric={len(LGBM_NUMERIC_FEATURES)}, categorical={LGBM_CATEGORICAL_FEATURES}')
print(f'LR Scorecard contract: {len(SCORECARD_FEATURES)} features')
print(f'  numeric={len(SCORECARD_NUMERIC_FEATURES)}, categorical={SCORECARD_CATEGORICAL_FEATURES}')
tables


## 1.1. Feature contract đầy đủ

Cell này lấy feature contract trực tiếp từ training scripts, map từng feature về cột Gold tương ứng, validate `gold.hc_features_v1`, và kiểm tra artifact đang load có đúng `feature_cols`/thứ tự feature hay không.


In [ ]:
# Bootstrap khi chạy cell trực tiếp sau kernel reset.
try:
    ROOT
except NameError:
    import sys
    from pathlib import Path

    def find_project_root() -> Path:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents, Path('/home/taitu/GitHub/Loan_ETL')]
        for candidate in candidates:
            if (candidate / 'machinelearning' / 'ml' / 'retrain_customer_model.py').exists():
                return candidate / 'machinelearning'
            if (candidate / 'data' / 'etl.duckdb').exists() or (candidate / 'ml' / 'retrain_customer_model.py').exists():
                return candidate
        return cwd

    ROOT = find_project_root()
    PROJECT_ROOT = ROOT.parent if ROOT.name == 'machinelearning' else ROOT
    sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from machinelearning.ml.feature_contract import (
    CUSTOMER_MODEL_NAME,
    SCORECARD_MODEL_NAME,
    build_feature_contract_frame,
    missing_gold_columns,
    validate_artifact_features,
)

try:
    con
except NameError:
    con = duckdb.connect(str(ROOT / 'data' / 'etl.duckdb'), read_only=True)

feature_contract = build_feature_contract_frame()

existing_gold_cols = set(con.execute("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema='gold' AND table_name='hc_features_v1'
""").df()['column_name'])
missing_cols = missing_gold_columns(existing_gold_cols)
if missing_cols:
    raise RuntimeError(
        'gold.hc_features_v1 thiếu cột cần cho model: ' + ', '.join(missing_cols)
    )

artifact_checks = []
for model_name, path in {
    CUSTOMER_MODEL_NAME: ROOT / 'ml' / 'models' / 'customer_risk_model.pkl',
    SCORECARD_MODEL_NAME: ROOT / 'ml' / 'models' / 'scorecard_model.pkl',
}.items():
    artifact = joblib.load(path)
    validate_artifact_features(model_name, artifact)
    artifact_checks.append({
        'model': model_name,
        'artifact': str(path.relative_to(ROOT)),
        'feature_count': len(artifact['feature_cols']),
        'status': 'feature_cols OK',
    })

summary = (
    feature_contract
    .groupby(['model', 'role'])
    .size()
    .reset_index(name='count')
)

artifact_df = pd.DataFrame(artifact_checks)
role_colors = {'numeric': '#3498DB', 'categorical': '#E67E22'}

fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 2.2], hspace=0.35, wspace=0.25)

# (1) Feature count by model/role
ax = fig.add_subplot(gs[0, 0])
pivot = summary.pivot(index='model', columns='role', values='count').fillna(0)
pivot[['numeric', 'categorical']].plot(
    kind='bar', stacked=True, ax=ax,
    color=[role_colors['numeric'], role_colors['categorical']],
    edgecolor='black', linewidth=0.4,
)
for container in ax.containers:
    ax.bar_label(container, label_type='center', color='white', fontweight='bold')
ax.set_title('Feature contract theo model', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Số feature')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Role')

# (2) Artifact feature count validation
ax = fig.add_subplot(gs[0, 1])
bars = ax.bar(
    artifact_df['model'], artifact_df['feature_count'],
    color=['#27AE60', '#8E44AD'], edgecolor='black', linewidth=0.5,
)
for bar, status in zip(bars, artifact_df['status']):
    value = int(bar.get_height())
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.5, f'{value}\n{status}',
            ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_ylim(0, max(artifact_df['feature_count']) + 6)
ax.set_title('Artifact feature_cols validation', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Số feature trong artifact')

def plot_feature_order(ax, model_name, title):
    data = feature_contract[feature_contract['model'] == model_name].copy()
    data = data.sort_values('position', ascending=False)
    colors = data['role'].map(role_colors)
    ax.barh(data['feature'], data['position'], color=colors, edgecolor='black', linewidth=0.35)
    for y, (_, row) in enumerate(data.iterrows()):
        source = row['gold_column']
        label = source if len(source) <= 28 else source[:25] + '...'
        ax.text(row['position'] + 0.35, y, label, va='center', fontsize=8, color='#2C3E50')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Vị trí trong feature_cols')
    ax.set_xlim(0, data['position'].max() + 10)
    ax.grid(axis='x', alpha=0.25)

# (3) Full LightGBM feature order + Gold source
plot_feature_order(fig.add_subplot(gs[1, 0]), CUSTOMER_MODEL_NAME, 'LightGBM: đủ 28 feature + source')

# (4) Full Scorecard feature order + Gold source
plot_feature_order(fig.add_subplot(gs[1, 1]), SCORECARD_MODEL_NAME, 'Scorecard: đủ 25 feature + source')

plt.tight_layout()
plt.show()


## 2. Đánh giá chất lượng dữ liệu

4 chart trong 1 hình:
- (top-left) Class balance — tỷ lệ default vs no-default
- (top-right) Null rate per feature ở Silver
- (bottom-left) Correlation với target cho numeric feature đang còn dùng ở Gold v3
- (bottom-right) Heatmap top numeric features

Gold v3 không còn dùng `ext_source_1/3`; các feature mới cần theo dõi là `years_employed` và `occupation_type`.


In [ ]:
# Bootstrap khi chạy cell trực tiếp sau kernel reset.
try:
    ROOT
except NameError:
    import sys
    from pathlib import Path

    def find_project_root() -> Path:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents, Path('/home/taitu/GitHub/Loan_ETL')]
        for candidate in candidates:
            if (candidate / 'machinelearning' / 'ml' / 'retrain_customer_model.py').exists():
                return candidate / 'machinelearning'
            if (candidate / 'data' / 'etl.duckdb').exists() or (candidate / 'ml' / 'retrain_customer_model.py').exists():
                return candidate
        return cwd

    ROOT = find_project_root()
    PROJECT_ROOT = ROOT.parent if ROOT.name == 'machinelearning' else ROOT
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    duckdb
except NameError:
    import duckdb
try:
    np
except NameError:
    import numpy as np
try:
    pd
except NameError:
    import pandas as pd
try:
    plt
except NameError:
    import matplotlib.pyplot as plt
try:
    sns
except NameError:
    import seaborn as sns
    sns.set_theme(style='whitegrid', context='notebook')

try:
    con.execute('SELECT 1')
except Exception:
    con = duckdb.connect(str(ROOT / 'data' / 'etl.duckdb'), read_only=True)


def require_gold_columns(columns):
    existing = set(con.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema='gold' AND table_name='hc_features_v1'
    """).df()['column_name'])
    missing = sorted(set(columns) - existing)
    if missing:
        raise RuntimeError(
            "gold.hc_features_v1 chưa được rebuild theo feature/model v3. "
            f"Thiếu cột: {', '.join(missing)}. "
            "Chạy từ project root: python -m machinelearning.etl.etl_silver && python -m machinelearning.etl.etl_gold"
        )

# Class balance
target_dist = con.execute(
    'SELECT is_default, COUNT(*) AS n FROM silver.home_credit_cleansed GROUP BY is_default ORDER BY is_default'
).df()

# Null rate per silver feature
silver_cols = con.execute(
    "SELECT column_name FROM information_schema.columns "
    "WHERE table_schema='silver' AND table_name='home_credit_cleansed'"
).df()['column_name'].tolist()
null_query = 'SELECT ' + ', '.join(
    f'ROUND(100.0 * AVG(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END), 2) AS "{c}"'
    for c in silver_cols
) + ' FROM silver.home_credit_cleansed'
null_df = con.execute(null_query).df().T.reset_index()
null_df.columns = ['feature', 'null_pct']
null_df = null_df.sort_values('null_pct', ascending=True).tail(15)

# Correlation với target trên các numeric Gold feature hiện còn dùng bởi model v3.
gold_numeric_cols = [
    'credit_score_midpoint', 'debt_to_income_ratio', 'loan_amount_to_income',
    'log_monthly_income', 'rating_ordinal', 'payment_to_income',
    'is_homeowner_flag', 'income_verifiable_flag', 'high_dti_flag',
    'num_previous_loans', 'previous_default_rate',
    'num_bureau_records', 'num_active_credit', 'total_overdue_amount',
    'max_credit_overdue_days', 'has_bad_debt',
    'years_employed', 'age_years', 'gender_male_flag', 'education_ordinal',
    'cnt_children', 'cnt_fam_members', 'is_married_flag', 'is_default',
]
require_gold_columns(gold_numeric_cols)
gold_df = con.execute(f"SELECT {', '.join(gold_numeric_cols)} FROM gold.hc_features_v1").df()
corr = gold_df.corr(numeric_only=True)['is_default'].drop('is_default').sort_values()
top_feats = corr.abs().sort_values(ascending=False).head(8).index.tolist() + ['is_default']
require_gold_columns(gold_numeric_cols)

# ── Plot 4-panel figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# (1) Class balance pie
axes[0, 0].pie(target_dist['n'],
               labels=[f"No Default\n{target_dist.iloc[0]['n']:,}",
                       f"Default\n{target_dist.iloc[1]['n']:,}"],
               colors=['#5DADE2', '#E74C3C'], autopct='%1.2f%%',
               wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ratio = target_dist.iloc[0]['n'] // target_dist.iloc[1]['n']
require_gold_columns(gold_numeric_cols)
axes[0, 0].set_title(f'Class balance — imbalance ~{ratio}:1', fontweight='bold')

# (2) Null rate top 15
colors = ['#7ED957' if v < 5 else '#F5B041' if v < 30 else '#E74C3C' for v in null_df['null_pct']]
require_gold_columns(gold_numeric_cols)
axes[0, 1].barh(null_df['feature'], null_df['null_pct'], color=colors, edgecolor='black', linewidth=0.4)
axes[0, 1].axvline(5,  color='#F5B041', linestyle='--', alpha=0.5)
axes[0, 1].axvline(30, color='#E74C3C', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Null rate (%)')
axes[0, 1].set_title('Null rate per feature (top 15)', fontweight='bold')

# (3) Correlation với target
corr_colors = ['#E74C3C' if v > 0 else '#5DADE2' for v in corr.values]
require_gold_columns(gold_numeric_cols)
axes[1, 0].barh(corr.index, corr.values, color=corr_colors, edgecolor='black', linewidth=0.4)
axes[1, 0].axvline(0, color='black', linewidth=0.8)
axes[1, 0].set_xlabel('Correlation với is_default')
axes[1, 0].set_title('Feature ↔ Target (đỏ: risk ↑, xanh: an toàn ↑)', fontweight='bold')

# (4) Heatmap top features
cm = gold_df[top_feats].corr(numeric_only=True)
mask = np.triu(np.ones_like(cm, dtype=bool), k=1)
sns.heatmap(cm, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=False,
            cbar_kws={'shrink': 0.8}, ax=axes[1, 1], annot_kws={'fontsize': 8})
axes[1, 1].set_title('Correlation heatmap — Top 8 numeric features', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n→ Imbalance {ratio}:1 — LightGBM dùng is_unbalance=True khi train')
print(f'→ Cột null cao nhất: {null_df.tail(3)[["feature","null_pct"]].to_dict(orient="records")}')
print(f'→ Numeric Gold features checked: {len(gold_numeric_cols) - 1}; categorical model features: employment_status + occupation_type')


## 3. EDA — Default rate theo các feature chính

4 chart trong 1 hình: credit score · DTI · years employed · occupation type

Các chart này bám theo feature hiện tại của LightGBM v3 và scorecard: credit/income burden, employment tenure, occupation group, cùng credit history/demographic ở phần correlation.


In [ ]:
# Bootstrap khi chạy cell trực tiếp sau kernel reset.
try:
    ROOT
except NameError:
    import sys
    from pathlib import Path

    def find_project_root() -> Path:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents, Path('/home/taitu/GitHub/Loan_ETL')]
        for candidate in candidates:
            if (candidate / 'machinelearning' / 'ml' / 'retrain_customer_model.py').exists():
                return candidate / 'machinelearning'
            if (candidate / 'data' / 'etl.duckdb').exists() or (candidate / 'ml' / 'retrain_customer_model.py').exists():
                return candidate
        return cwd

    ROOT = find_project_root()
    PROJECT_ROOT = ROOT.parent if ROOT.name == 'machinelearning' else ROOT
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    duckdb
except NameError:
    import duckdb
try:
    np
except NameError:
    import numpy as np
try:
    pd
except NameError:
    import pandas as pd
try:
    plt
except NameError:
    import matplotlib.pyplot as plt
try:
    sns
except NameError:
    import seaborn as sns
    sns.set_theme(style='whitegrid', context='notebook')

try:
    con.execute('SELECT 1')
except Exception:
    con = duckdb.connect(str(ROOT / 'data' / 'etl.duckdb'), read_only=True)


def require_gold_columns(columns):
    existing = set(con.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema='gold' AND table_name='hc_features_v1'
    """).df()['column_name'])
    missing = sorted(set(columns) - existing)
    if missing:
        raise RuntimeError(
            "gold.hc_features_v1 chưa được rebuild theo feature/model v3. "
            f"Thiếu cột: {', '.join(missing)}. "
            "Chạy từ project root: python -m machinelearning.etl.etl_silver && python -m machinelearning.etl.etl_gold"
        )

# Tính 4 bảng aggregated
require_gold_columns(['credit_score_midpoint', 'debt_to_income_ratio', 'years_employed', 'occupation_type', 'is_default'])
score_df = con.execute('''
    SELECT credit_score_midpoint AS score, is_default
    FROM gold.hc_features_v1
    WHERE credit_score_midpoint IS NOT NULL
''').df()

dti_df = con.execute('''
    SELECT
        CASE
          WHEN debt_to_income_ratio < 0.1 THEN '< 0.1'
          WHEN debt_to_income_ratio < 0.2 THEN '0.1–0.2'
          WHEN debt_to_income_ratio < 0.3 THEN '0.2–0.3'
          WHEN debt_to_income_ratio < 0.5 THEN '0.3–0.5'
          WHEN debt_to_income_ratio < 1.0 THEN '0.5–1.0'
          ELSE '> 1.0'
        END AS bucket,
        ROUND(AVG(is_default) * 100, 2) AS default_pct,
        COUNT(*) AS n
    FROM gold.hc_features_v1
    WHERE debt_to_income_ratio IS NOT NULL
    GROUP BY bucket
    ORDER BY MIN(debt_to_income_ratio)
''').df()

tenure_df = con.execute('''
    SELECT
        CASE
          WHEN years_employed < 1 THEN '< 1y'
          WHEN years_employed < 3 THEN '1–3y'
          WHEN years_employed < 5 THEN '3–5y'
          WHEN years_employed < 10 THEN '5–10y'
          WHEN years_employed < 20 THEN '10–20y'
          ELSE '20y+'
        END AS bucket,
        ROUND(AVG(is_default) * 100, 2) AS default_pct,
        COUNT(*) AS n
    FROM gold.hc_features_v1
    WHERE years_employed IS NOT NULL
    GROUP BY bucket
    ORDER BY MIN(years_employed)
''').df()

occupation_df = con.execute('''
    SELECT
        occupation_type,
        ROUND(AVG(is_default) * 100, 2) AS default_pct,
        COUNT(*) AS n
    FROM gold.hc_features_v1
    WHERE occupation_type IS NOT NULL
    GROUP BY occupation_type
    HAVING COUNT(*) >= 500
    ORDER BY default_pct DESC
    LIMIT 10
''').df().sort_values('default_pct')

# ── Plot 4-panel figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# (1) Credit score density by class
sns.histplot(data=score_df, x='score', hue='is_default', bins=40,
             palette=['#5DADE2', '#E74C3C'], alpha=0.7,
             stat='density', common_norm=False, ax=axes[0, 0])
axes[0, 0].set_title('Phân phối credit score theo class', fontweight='bold')
axes[0, 0].set_xlabel('Credit Score')

# (2) DTI buckets
bars = axes[0, 1].bar(dti_df['bucket'], dti_df['default_pct'], color='#E74C3C',
                       alpha=0.75, edgecolor='black')
axes[0, 1].set_title('Default rate theo DTI bucket', fontweight='bold')
axes[0, 1].set_ylabel('Default rate (%)')
for b, v in zip(bars, dti_df['default_pct']):
    axes[0, 1].text(b.get_x() + b.get_width()/2, v + 0.2, f'{v}%',
                    ha='center', fontweight='bold', fontsize=9)

# (3) Years employed buckets
bars = axes[1, 0].bar(tenure_df['bucket'], tenure_df['default_pct'], color='#3498DB',
                       alpha=0.75, edgecolor='black')
axes[1, 0].set_title('Default rate theo thâm niên làm việc', fontweight='bold')
axes[1, 0].set_ylabel('Default rate (%)')
for b, v in zip(bars, tenure_df['default_pct']):
    axes[1, 0].text(b.get_x() + b.get_width()/2, v + 0.1, f'{v}%',
                    ha='center', fontweight='bold', fontsize=9)

# (4) Occupation type
bars = axes[1, 1].barh(occupation_df['occupation_type'], occupation_df['default_pct'],
                        color='#16A085', alpha=0.75, edgecolor='black')
axes[1, 1].set_title('Top occupation groups theo default rate', fontweight='bold')
axes[1, 1].set_xlabel('Default rate (%)')
for b, v in zip(bars, occupation_df['default_pct']):
    axes[1, 1].text(v + 0.1, b.get_y() + b.get_height()/2, f'{v}%',
                    va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()


## 4. Train models

Train cả 2 model trực tiếp từ notebook:
- `customer_lgbm_v3`: 28 feature, LightGBM, `is_unbalance=True`, 2 categorical feature qua `OrdinalEncoder`
- `scorecard_model.pkl`: 25 feature, Logistic Regression scorecard, FICO-style 300–850

Cell này ghi đè artifact trong `machinelearning/ml/models/`.


In [ ]:
# Bootstrap khi chạy cell trực tiếp sau kernel reset.
try:
    ROOT
except NameError:
    import sys
    from pathlib import Path

    def find_project_root() -> Path:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents, Path('/home/taitu/GitHub/Loan_ETL')]
        for candidate in candidates:
            if (candidate / 'machinelearning' / 'ml' / 'retrain_customer_model.py').exists():
                return candidate / 'machinelearning'
            if (candidate / 'data' / 'etl.duckdb').exists() or (candidate / 'ml' / 'retrain_customer_model.py').exists():
                return candidate
        return cwd

    ROOT = find_project_root()
    PROJECT_ROOT = ROOT.parent if ROOT.name == 'machinelearning' else ROOT
    sys.path.insert(0, str(PROJECT_ROOT))

# DuckDB không cho phép cùng 1 file mở với 2 mode (read-only vs read-write) trong cùng process.
# Đóng `con` (read-only) trước khi train (cần read-write qua SQLAlchemy).
try:
    con.close()
except Exception:
    pass

from machinelearning.ml.retrain_customer_model import train as train_lgbm
from machinelearning.ml.train_scorecard       import train as train_scorecard
from machinelearning.utils.db_connection      import reset_engine

try:
    train_lgbm()
    print('\n' + '='*60 + '\n')
    train_scorecard()
finally:
    reset_engine()

# Lưu ý: sau cell này, `con` đã đóng. Nếu muốn dùng `con` lại → restart kernel + chạy từ đầu.


## 5. Đánh giá Model

Load 2 artifact đã train, build lại test set theo đúng preprocessing hiện tại, đánh giá 6 chỉ tiêu trong 1 figure:
- ROC curve · Confusion matrix · Feature importance · Score distribution · Calibration · Precision-Recall

Evaluation dùng cùng query và feature list với `machinelearning/ml/retrain_customer_model.py` và `machinelearning/ml/train_scorecard.py`, tránh lệch contract khi feature thay đổi.


In [ ]:
# Bootstrap khi chạy cell trực tiếp sau kernel reset.
try:
    ROOT
except NameError:
    import sys
    from pathlib import Path

    def find_project_root() -> Path:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents, Path('/home/taitu/GitHub/Loan_ETL')]
        for candidate in candidates:
            if (candidate / 'machinelearning' / 'ml' / 'retrain_customer_model.py').exists():
                return candidate / 'machinelearning'
            if (candidate / 'data' / 'etl.duckdb').exists() or (candidate / 'ml' / 'retrain_customer_model.py').exists():
                return candidate
        return cwd

    ROOT = find_project_root()
    PROJECT_ROOT = ROOT.parent if ROOT.name == 'machinelearning' else ROOT
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    duckdb
except NameError:
    import duckdb
try:
    np
except NameError:
    import numpy as np
try:
    pd
except NameError:
    import pandas as pd
try:
    plt
except NameError:
    import matplotlib.pyplot as plt
try:
    sns
except NameError:
    import seaborn as sns
    sns.set_theme(style='whitegrid', context='notebook')

from machinelearning.ml.train_scorecard import (
    NUMERIC_FEATURES as SCORECARD_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES as SCORECARD_CATEGORICAL_FEATURES,
)

import joblib
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve

from machinelearning.ml.retrain_customer_model import (
    _QUERY as LGBM_QUERY,
    ALL_FEATURES as LGBM_FEATURES,
    NUMERIC_FEATURES as LGBM_NUMERIC_FEATURES,
)
from machinelearning.ml.train_scorecard import QUERY as SCORE_QUERY, ALL_FEATURES as SCORE_FEATURES, prob_to_score

lgbm_art      = joblib.load(ROOT / 'ml' / 'models' / 'customer_risk_model.pkl')
scorecard_art = joblib.load(ROOT / 'ml' / 'models' / 'scorecard_model.pkl')

# Evaluation chỉ đọc dữ liệu; dispose SQLAlchemy engine write-mode trước khi mở read-only.
from machinelearning.utils.db_connection import reset_engine
reset_engine()

try:
    con.close()
except Exception:
    pass
eval_con = duckdb.connect(str(ROOT / 'data' / 'etl.duckdb'), read_only=True)

existing_gold_cols = set(eval_con.execute("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema='gold' AND table_name='hc_features_v1'
""").df()['column_name'])
missing_gold_cols = sorted({'years_employed', 'occupation_type'} - existing_gold_cols)
if missing_gold_cols:
    raise RuntimeError(
        "gold.hc_features_v1 chưa được rebuild theo feature/model v3. "
        f"Thiếu cột: {', '.join(missing_gold_cols)}. "
        "Chạy từ project root: python -m machinelearning.etl.etl_silver && python -m machinelearning.etl.etl_gold"
    )


# ── Test set cho LightGBM v3 ─────────────────────────────────────────────
df_l = eval_con.execute(LGBM_QUERY).df()
df_l['employment_status'] = df_l['employment_status'].fillna('Other/Unknown')
df_l['occupation_type']   = df_l['occupation_type'].fillna('Unknown')
# DuckDB can return pandas nullable Int32 with pd.NA; convert to float64/np.nan for sklearn.
df_l[LGBM_NUMERIC_FEATURES] = df_l[LGBM_NUMERIC_FEATURES].astype('float64')
# LightGBM handles NaN natively for numeric features; match training cleanup.
df_l = df_l.dropna(subset=['is_default', 'monthly_income', 'loan_amount', 'credit_score'])
_, Xl_te, _, yl_te = train_test_split(
    df_l[LGBM_FEATURES], df_l['is_default'],
    test_size=0.2, random_state=42, stratify=df_l['is_default']
)
y_prob_lgbm = lgbm_art['pipeline'].predict_proba(Xl_te)[:, 1]
lgbm_threshold = float(lgbm_art.get('thresholds', {}).get('high', 0.4))

# ── Test set cho LR Scorecard ───────────────────────────────────────────
df_s = eval_con.execute(SCORE_QUERY).df()
eval_con.close()
df_s[SCORECARD_NUMERIC_FEATURES] = df_s[SCORECARD_NUMERIC_FEATURES].fillna(
    df_s[SCORECARD_NUMERIC_FEATURES].median()
)
df_s[SCORECARD_CATEGORICAL_FEATURES] = df_s[SCORECARD_CATEGORICAL_FEATURES].fillna('Other/Unknown')
df_s = df_s.dropna(subset=['is_default'])
_, Xs_te, _, ys_te = train_test_split(
    df_s[SCORE_FEATURES], df_s['is_default'],
    test_size=0.2, random_state=42, stratify=df_s['is_default']
)
y_prob_score = scorecard_art['pipeline'].predict_proba(Xs_te)[:, 1]

auc_lgbm  = roc_auc_score(yl_te, y_prob_lgbm)
auc_score = roc_auc_score(ys_te, y_prob_score)
print(f'LightGBM ({lgbm_art.get("model_version", "unknown")}) AUC = {auc_lgbm:.4f}')
print(f'LR Scorecard AUC = {auc_score:.4f}')
print(f'LightGBM features: {len(lgbm_art["feature_cols"])} | Scorecard features: {len(scorecard_art["feature_cols"])}')

# ── Plot 6-panel figure ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# (1) ROC
fpr_l, tpr_l, _ = roc_curve(yl_te, y_prob_lgbm)
fpr_s, tpr_s, _ = roc_curve(ys_te, y_prob_score)
axes[0, 0].plot(fpr_l, tpr_l, color='#27AE60', linewidth=2.5, label=f'LightGBM ({auc_lgbm:.3f})')
axes[0, 0].plot(fpr_s, tpr_s, color='#8E44AD', linewidth=2.5, label=f'LR Score ({auc_score:.3f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0, 0].set_xlabel('FPR'); axes[0, 0].set_ylabel('TPR')
axes[0, 0].set_title('ROC Curve', fontweight='bold'); axes[0, 0].legend(loc='lower right')

# (2) Confusion matrix - LightGBM high-risk threshold
cm_l = confusion_matrix(yl_te, (y_prob_lgbm >= lgbm_threshold).astype(int))
sns.heatmap(cm_l, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0, 1],
            xticklabels=['Pred 0', 'Pred 1'], yticklabels=['Act 0', 'Act 1'],
            annot_kws={'fontsize': 13, 'fontweight': 'bold'})
tn, fp, fn, tp = cm_l.ravel()
recall = tp / (tp + fn) if (tp + fn) else 0
axes[0, 1].set_title(f'Confusion Matrix — LightGBM @ {lgbm_threshold:.2f}\nRecall(default) = {recall:.2%}',
                     fontweight='bold')

# (3) Feature importance
lgbm_clf = lgbm_art['pipeline'].named_steps['classifier']
importance_names = lgbm_art.get('feature_names_out') or LGBM_FEATURES
importance_names = [name.replace('num__', '').replace('cat__', '') for name in importance_names]
imp = pd.DataFrame({'feat': importance_names, 'imp': lgbm_clf.feature_importances_})
imp = imp.sort_values('imp', ascending=True).tail(15)
axes[0, 2].barh(imp['feat'], imp['imp'], color='#16A085', edgecolor='black', linewidth=0.4)
axes[0, 2].set_xlabel('Split count'); axes[0, 2].set_title('Top 15 Feature Importance (LightGBM)', fontweight='bold')

# (4) Score distribution
test_scores = prob_to_score(y_prob_score)
score_dist  = pd.DataFrame({'score': test_scores, 'is_default': ys_te.values})
sns.histplot(data=score_dist, x='score', hue='is_default', bins=40,
             palette=['#5DADE2', '#E74C3C'], alpha=0.7,
             stat='density', common_norm=False, ax=axes[1, 0])
axes[1, 0].set_title(f'Credit Score Distribution\n[{test_scores.min()}–{test_scores.max()}], mean = {test_scores.mean():.0f}',
                     fontweight='bold')
axes[1, 0].set_xlabel('Credit Score (300–850)')

# (5) Calibration
fp_l, mp_l = calibration_curve(yl_te, y_prob_lgbm,  n_bins=10, strategy='quantile')
fp_s, mp_s = calibration_curve(ys_te, y_prob_score, n_bins=10, strategy='quantile')
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect')
axes[1, 1].plot(mp_l, fp_l, 'o-', color='#27AE60', linewidth=2, markersize=8, label='LightGBM')
axes[1, 1].plot(mp_s, fp_s, 's-', color='#8E44AD', linewidth=2, markersize=8, label='LR Score')
axes[1, 1].set_xlabel('Predicted prob'); axes[1, 1].set_ylabel('Actual rate')
axes[1, 1].set_title('Calibration Plot', fontweight='bold'); axes[1, 1].legend(loc='upper left')

# (6) Precision-Recall
pr_l, rc_l, _ = precision_recall_curve(yl_te, y_prob_lgbm)
pr_s, rc_s, _ = precision_recall_curve(ys_te, y_prob_score)
ap_l = average_precision_score(yl_te, y_prob_lgbm)
ap_s = average_precision_score(ys_te, y_prob_score)
axes[1, 2].plot(rc_l, pr_l, color='#27AE60', linewidth=2.5, label=f'LightGBM (AP={ap_l:.3f})')
axes[1, 2].plot(rc_s, pr_s, color='#8E44AD', linewidth=2.5, label=f'LR Score (AP={ap_s:.3f})')
axes[1, 2].axhline(yl_te.mean(), color='k', linestyle='--', alpha=0.4,
                   label=f'Baseline ({yl_te.mean():.3f})')
axes[1, 2].set_xlabel('Recall'); axes[1, 2].set_ylabel('Precision')
axes[1, 2].set_title('Precision-Recall Curve', fontweight='bold'); axes[1, 2].legend(loc='upper right')

plt.tight_layout()
plt.show()


## 6. Inference mẫu với đầy đủ feature

Cell này minh họa cách gọi cả hai artifact bằng một hồ sơ mẫu có đủ feature. Điểm quan trọng là DataFrame cuối cùng luôn được reorder theo `artifact['feature_cols']`, giống backend production.


In [ ]:
# Bootstrap khi chạy cell trực tiếp sau kernel reset.
try:
    ROOT
except NameError:
    import sys
    from pathlib import Path

    def find_project_root() -> Path:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents, Path('/home/taitu/GitHub/Loan_ETL')]
        for candidate in candidates:
            if (candidate / 'machinelearning' / 'ml' / 'retrain_customer_model.py').exists():
                return candidate / 'machinelearning'
            if (candidate / 'data' / 'etl.duckdb').exists() or (candidate / 'ml' / 'retrain_customer_model.py').exists():
                return candidate
        return cwd

    ROOT = find_project_root()
    PROJECT_ROOT = ROOT.parent if ROOT.name == 'machinelearning' else ROOT
    sys.path.insert(0, str(PROJECT_ROOT))

import math
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from machinelearning.ml.feature_contract import (
    CUSTOMER_MODEL_NAME,
    SCORECARD_MODEL_NAME,
    validate_artifact_features,
)
from machinelearning.ml.train_scorecard import prob_to_score

lgbm_art      = joblib.load(ROOT / 'ml' / 'models' / 'customer_risk_model.pkl')
scorecard_art = joblib.load(ROOT / 'ml' / 'models' / 'scorecard_model.pkl')
validate_artifact_features(CUSTOMER_MODEL_NAME, lgbm_art)
validate_artifact_features(SCORECARD_MODEL_NAME, scorecard_art)

sample_app = {
    'monthly_income': 5000.0,
    'loan_amount': 120000.0,
    'term': 36,
    'employment_status': 'Employed',
    'is_homeowner': 1,
    'listing_category': 1,
    'credit_score': 720,
    'occupation_type': 'Core staff',
    'years_employed': 6.0,
    'num_previous_loans': 2,
    'previous_default_rate': 0.0,
    'num_bureau_records': 4,
    'num_active_credit': 2,
    'total_overdue_amount': 0.0,
    'max_credit_overdue_days': 0,
    'has_bad_debt': 0,
    'income_verifiable_flag': 1,
    'age_years': 34,
    'gender_male_flag': 1,
    'education_ordinal': 4,
    'cnt_children': 1,
    'cnt_fam_members': 3,
    'is_married_flag': 1,
}

def rating_ordinal(credit_score: int) -> int:
    if credit_score >= 760: return 7
    if credit_score >= 700: return 6
    if credit_score >= 660: return 5
    if credit_score >= 620: return 4
    if credit_score >= 580: return 3
    if credit_score >= 540: return 2
    return 1

def ordered_row(values: dict, feature_cols: list[str]) -> pd.DataFrame:
    missing = [feature for feature in feature_cols if feature not in values]
    if missing:
        raise ValueError('Thiếu feature: ' + ', '.join(missing))
    return pd.DataFrame([{feature: values[feature] for feature in feature_cols}], columns=feature_cols)

mi = sample_app['monthly_income']
la = sample_app['loan_amount']
term = sample_app['term']
credit_score = sample_app['credit_score']
dti = 0.32
hc_payment_to_income = (la / term) / mi if mi > 0 and term > 0 else 0.0

customer_features = {
    **sample_app,
    'dti': dti,
    'log_monthly_income': math.log1p(max(mi, 0)),
    'loan_amount_to_income': la / mi if mi else 0.0,
    'rating_ordinal': rating_ordinal(credit_score),
    'high_dti_flag': int(dti > float(lgbm_art.get('dti_p75', 0.4))),
}

scorecard_features = {
    'credit_score_midpoint': credit_score,
    'debt_to_income_ratio': hc_payment_to_income,
    'loan_amount_to_income': la / (mi * 12) if mi else 0.0,
    'log_monthly_income': math.log1p(max(mi, 0)),
    'rating_ordinal': rating_ordinal(credit_score),
    'is_homeowner_flag': sample_app['is_homeowner'],
    'income_verifiable_flag': sample_app['income_verifiable_flag'],
    'high_dti_flag': int(hc_payment_to_income > float(scorecard_art.get('dti_p75', 2.683))),
    'payment_to_income': hc_payment_to_income,
    'num_previous_loans': sample_app['num_previous_loans'],
    'previous_default_rate': sample_app['previous_default_rate'],
    'num_bureau_records': sample_app['num_bureau_records'],
    'num_active_credit': sample_app['num_active_credit'],
    'total_overdue_amount': sample_app['total_overdue_amount'],
    'max_credit_overdue_days': sample_app['max_credit_overdue_days'],
    'has_bad_debt': sample_app['has_bad_debt'],
    'years_employed': sample_app['years_employed'],
    'age_years': sample_app['age_years'],
    'gender_male_flag': sample_app['gender_male_flag'],
    'education_ordinal': sample_app['education_ordinal'],
    'cnt_children': sample_app['cnt_children'],
    'cnt_fam_members': sample_app['cnt_fam_members'],
    'is_married_flag': sample_app['is_married_flag'],
    'employment_status_grouped': sample_app['employment_status'],
    'occupation_type': sample_app['occupation_type'],
}

customer_row = ordered_row(customer_features, lgbm_art['feature_cols'])
scorecard_row = ordered_row(scorecard_features, scorecard_art['feature_cols'])

customer_pd = float(lgbm_art['pipeline'].predict_proba(customer_row)[0, 1])
scorecard_pd = float(scorecard_art['pipeline'].predict_proba(scorecard_row)[0, 1])
credit_score_fico = int(prob_to_score([scorecard_pd])[0])

prediction_summary = pd.DataFrame([
    {
        'model': CUSTOMER_MODEL_NAME,
        'feature_count': customer_row.shape[1],
        'p_default': round(customer_pd, 4),
        'risk_level': 'Low' if customer_pd < lgbm_art['thresholds']['low'] else 'High' if customer_pd > lgbm_art['thresholds']['high'] else 'Medium',
    },
    {
        'model': SCORECARD_MODEL_NAME,
        'feature_count': scorecard_row.shape[1],
        'p_default': round(scorecard_pd, 4),
        'credit_score': credit_score_fico,
    },
])

fig = plt.figure(figsize=(18, 13))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 2.4], hspace=0.35, wspace=0.3)

# (1) Default probability by model
ax = fig.add_subplot(gs[0, 0])
prob_df = prediction_summary[['model', 'p_default']]
bars = ax.bar(prob_df['model'], prob_df['p_default'], color=['#27AE60', '#8E44AD'],
              edgecolor='black', linewidth=0.5)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2%}', ha='center', va='bottom', fontweight='bold')
ax.axhline(lgbm_art['thresholds']['low'], color='#F1C40F', linestyle='--', linewidth=1.5,
           label=f"Low threshold {lgbm_art['thresholds']['low']:.2f}")
ax.axhline(lgbm_art['thresholds']['high'], color='#E74C3C', linestyle='--', linewidth=1.5,
           label=f"High threshold {lgbm_art['thresholds']['high']:.2f}")
ax.set_ylim(0, max(0.45, prob_df['p_default'].max() + 0.08))
ax.set_title('Sample P(default)', fontweight='bold')
ax.set_ylabel('Probability')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper right')

# (2) Credit score output
ax = fig.add_subplot(gs[0, 1])
ax.barh(['Scorecard'], [credit_score_fico], color='#5DADE2', edgecolor='black', linewidth=0.5)
ax.set_xlim(300, 850)
ax.axvspan(300, 579, color='#E74C3C', alpha=0.16)
ax.axvspan(580, 669, color='#F39C12', alpha=0.16)
ax.axvspan(670, 739, color='#F1C40F', alpha=0.16)
ax.axvspan(740, 850, color='#27AE60', alpha=0.16)
ax.text(credit_score_fico + 8, 0, str(credit_score_fico), va='center', fontweight='bold', fontsize=13)
ax.set_title('Sample FICO-style score', fontweight='bold')
ax.set_xlabel('Credit score')

def plot_feature_payload(ax, row, title, categorical_features):
    payload = row.T.reset_index()
    payload.columns = ['feature', 'value']
    payload['position'] = range(1, len(payload) + 1)
    payload['role'] = payload['feature'].apply(
        lambda feature: 'categorical' if feature in categorical_features else 'numeric'
    )
    payload = payload.sort_values('position', ascending=False)
    colors = payload['role'].map({'numeric': '#3498DB', 'categorical': '#E67E22'})
    ax.barh(payload['feature'], [1] * len(payload), color=colors, edgecolor='black', linewidth=0.35)
    for y, (_, item) in enumerate(payload.iterrows()):
        value = item['value']
        if isinstance(value, float):
            label = f'{value:.4g}'
        else:
            label = str(value)
        ax.text(1.03, y, label, va='center', fontsize=8, color='#2C3E50')
    ax.set_xlim(0, 1.45)
    ax.set_xticks([])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Mỗi dòng = 1 feature được truyền vào model')

# (3) Full LightGBM feature payload
plot_feature_payload(
    fig.add_subplot(gs[1, 0]), customer_row,
    f'LightGBM payload: {customer_row.shape[1]}/{len(lgbm_art["feature_cols"])} feature',
    {'employment_status', 'occupation_type'},
)

# (4) Full Scorecard feature payload
plot_feature_payload(
    fig.add_subplot(gs[1, 1]), scorecard_row,
    f'Scorecard payload: {scorecard_row.shape[1]}/{len(scorecard_art["feature_cols"])} feature',
    {'employment_status_grouped', 'occupation_type'},
)

plt.tight_layout()
plt.show()


## Kết luận

**Data / Feature layer:**
- Gold source chính là `gold.hc_features_v1`, sinh từ Silver + aggregate `bureau` và `previous_application`.
- Feature hiện tại đã bỏ `ext_source_1/3` để tránh phụ thuộc điểm bên thứ ba không có tại inference.
- Feature mới trong v3: `years_employed` và `occupation_type`; nhóm demographic/bureau/history hiện được yêu cầu từ form hoặc DB history.

**Models:**
- `customer_lgbm_v3`: LightGBM với 28 feature = 22 input từ người dùng + 4 feature tự tính + 2 feature lịch sử DB; dùng cho `/applications/evaluate` và `/applications/submit`.
- `scorecard_model.pkl`: Logistic Regression scorecard với 25 feature, quy đổi probability sang FICO-style score 300–850 cho `/credit-score/{id}`.
- Threshold nghiệp vụ hiện tại: `low=0.20`, `high=0.40`; `risk_score = round((1 - P(default)) * 100)` trong backend.

**Điểm cần theo dõi:**
- AUC/PR nên đọc từ output cell evaluation sau mỗi lần retrain, không hard-code trong markdown.
- `occupation_type` là categorical feature; nếu frontend gửi giá trị ngoài danh mục training, encoder map về `-1`.
- `imputed_features` của customer model v3 luôn rỗng vì các input chính đã required ở schema/backend.
